## 8. 用 LangChain 实现混合检索

使用 Dense、BM25 和 RRF 完成混合检索与答案生成。

### 8.1. 先看一个混合检索例子

用户问：**“冲锋衣仅喷涂 DWR 涂层、未做全压胶处理，Listing 标题能否宣传为 100% Waterproof？”**

知识库里同时存在两类相近资料：

- 产品质检报告：包含 DWR、Waterproof 和测试结果。
- FTC 法规：明确规定只有 DWR、没有全压胶时，不得标注 `100% Waterproof`。

Dense 按语义相似度检索，容易把同样包含 DWR 和 Waterproof 的产品质检报告排在前面；BM25 对问题中的 `DWR`、`100% Waterproof` 等精确词更敏感，会把法规条款排在前面。

RRF（Reciprocal Rank Fusion，倒数排名融合）把每个检索器看成一位评委：评委只给候选片段排名。名次越靠前，得分越高；多位评委都认为某个片段靠前，它最终就靠前。

原始公式：

$$
\operatorname{RRF}(d)=\sum_{i=1}^{N}\frac{1}{c+\operatorname{rank}_i(d)}
$$

`rank_i(d)` 是片段 $d$ 在第 $i$ 路结果中的名次。`c=60` 用来缩小相邻名次的分差，避免某一路第一名一票定胜负。

这个问题的关键不是找到“冲锋衣防水”资料，而是找到“只有 DWR 且没有全压胶时能否这样宣传”的法规结论。

### 8.1.1. 加载文档与模型

Embedding 和 BM25 检索 `page_content`。将缺失的父级标题补入正文，使 SKU 和产品名参与检索；标题同时保留在 `metadata` 中，用于展示、过滤和定位片段。

In [1]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

load_dotenv("../.env")
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")

embeddings = HuggingFaceEmbeddings(
    model=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

data_dir = Path("../data")
documents = []
for path in sorted(data_dir.rglob("*.md")):
    if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
        continue
    documents.append(
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={"source": path.relative_to(data_dir).as_posix()},
        )
    )

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "标题"), ("##", "章节"), ("###", "小节")],
    strip_headers=False,
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=0)
chunks = []
for document in documents:
    header_chunks = header_splitter.split_text(document.page_content)
    for chunk in header_chunks:
        chunk.metadata.update(document.metadata)
    chunks.extend(text_splitter.split_documents(header_chunks))
searchable_chunks = []
for chunk_id, chunk in enumerate(chunks):
    headings = [
        chunk.metadata.get("标题"),
        chunk.metadata.get("章节"),
        chunk.metadata.get("小节"),
    ]
    missing_headings = [heading for heading in headings if heading and heading not in chunk.page_content]
    searchable_text = "\n".join([*missing_headings, chunk.page_content])
    searchable_chunks.append(
        Document(
            page_content=searchable_text,
            metadata={**chunk.metadata, "chunk_id": chunk_id},
        )
    )

print(f"加载文档：{len(documents)} 篇，分块：{len(searchable_chunks)} 段")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

加载文档：14 篇，分块：163 段


### 8.1.2. 构建 Dense 与 BM25 检索器

BM25 的 `preprocess_func` 保留完整的英文、数字、SKU 和型号词，并将中文按单字切分。

两路检索各返回 10 个候选，再由 RRF 融合。候选数过小会限制召回，过大则增加计算量和噪声。此处的 `k` 是检索候选数，与生成参数 `top_k` 无关。

In [2]:
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="`langchain-community` is being sunset.*")
    from langchain_community.retrievers import BM25Retriever
from langchain_qdrant import QdrantVectorStore

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_#-]+|[\u4e00-\u9fff]", text.lower())

vector_store = QdrantVectorStore.from_documents(
    documents=searchable_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="fashion_knowledge_hybrid",
)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(searchable_chunks, preprocess_func=tokenize)
bm25_retriever.k = 10

### 8.1.3. 对比并融合检索排名

`EnsembleRetriever` 按加权 RRF 合并 Dense 与 BM25 的排名：

$$
\operatorname{score}(d)=\sum_{i=1}^{N}\frac{w_i}{c+\operatorname{rank}_i(d)}
$$

`weights=[0.5, 0.5]` 表示两路权重相同，`id_key="chunk_id"` 用于合并同一片段。RRF 按排名融合，不验证内容真实性。

In [3]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5],
    c=60,
    id_key="chunk_id",
)

question = "冲锋衣仅喷涂 DWR 涂层、未做全压胶处理，Listing 标题能否宣传为 100% Waterproof？"
dense_results = dense_retriever.invoke(question)
bm25_results = bm25_retriever.invoke(question)
hybrid_results = hybrid_retriever.invoke(question)


def show_results(name, results, limit=5):
    print(f"\n{name} Top-{limit}")
    for rank, doc in enumerate(results[:limit], start=1):
        preview = doc.page_content.replace("\n", " ")[:90]
        print(f"{rank}. {doc.metadata['source']}｜片段 {doc.metadata['chunk_id']}｜{preview}")


print(f"查询：{question}")
show_results("Dense", dense_results)
show_results("BM25", bm25_results)
show_results("RRF", hybrid_results)
assert dense_results[0].metadata["source"] == "产品/冲锋衣-JK902/质检报告.md"
assert bm25_results[0].metadata["source"] == "法规政策/FTC纺织标识法.md"
assert hybrid_results[0].metadata["source"] == "法规政策/FTC纺织标识法.md"
assert "不得标" in hybrid_results[0].page_content


查询：冲锋衣仅喷涂 DWR 涂层、未做全压胶处理，Listing 标题能否宣传为 100% Waterproof？

Dense Top-5
1. 产品/冲锋衣-JK902/质检报告.md｜片段 25｜SGS 质检报告 - 冲锋衣 SKU-JK902 ## 3. 结论与签章   12项测试全部通过。SKU-JK902冲锋衣符合FTC Waterproof全防水标称条件。   | 
2. 法规政策/FTC纺织标识法.md｜片段 147｜美国 FTC 纺织品标识法与防水宣传合规禁令 4. 标签与宣称规则 ### 4.2 禁止行为   1. 仅 DWR 处理的面料 (无全压胶) 不得标 "100% Waterproo
3. 产品/冲锋衣-JK902/质检报告.md｜片段 24｜SGS 质检报告 - 冲锋衣 SKU-JK902 2. 详细测试结果 ### 2.6 DWR防泼水 (AATCC 22)   | 条件 | 评分(满分100) | 判定 | | :
4. 产品/冲锋衣-JK902/产品规格.md｜片段 4｜冲锋衣 SKU-JK902 技术规格书 2. 面料结构 ### 2.3 透湿性能   - 透湿率: ≥18,000 g/m²/24h (JIS L1099 B1) - 原理: eP
5. 产品/冲锋衣-JK902/质检报告.md｜片段 18｜SGS 质检报告 - 冲锋衣 SKU-JK902 ## 1. 测试摘要   | 序号 | 测试项目 | 测试方法 | 判定标准 | 结果 | 判定 | | :---: | :---

BM25 Top-5
1. 法规政策/FTC纺织标识法.md｜片段 147｜美国 FTC 纺织品标识法与防水宣传合规禁令 4. 标签与宣称规则 ### 4.2 禁止行为   1. 仅 DWR 处理的面料 (无全压胶) 不得标 "100% Waterproo
2. 法规政策/FTC纺织标识法.md｜片段 144｜美国 FTC 纺织品标识法与防水宣传合规禁令 3. 术语定义 ### 3.2 Water-Resistant (抗水)   指面料经过 DWR (Durable Water Rep
3. 法规政策/FTC纺织标识法.md｜片段 152｜美国 FTC 纺织品标识法与防水宣传合规禁令 ## 6. 合规检查清单   在 Listing 上线前, 

### 8.2. 使用混合检索结果生成答案

取 RRF 排名前 4 的片段作为上下文，保留来源信息，再交给模型生成答案。

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


def format_docs(docs):
    return "\n\n".join(
        f"[来源：{doc.metadata['source']}]\n{doc.page_content}" for doc in docs
    )


answer_docs = hybrid_results[:4]
context = format_docs(answer_docs)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是服饰箱包知识库助手。只根据提供的资料回答，并在答案末尾列出依据的来源；资料没有答案时回答‘现有资料无法回答’，不要猜测。"),
        ("human", "资料：\n{context}\n\n问题：{question}"),
    ]
)

api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
model = os.getenv("LLM_MODEL")
if api_key and model:
    llm = ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=os.getenv("LLM_BASE_URL") or None,
        temperature=0,
    )
    answer_chain = prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({"context": context, "question": question})
    print(f"问题：{question}\n\n答案：\n{answer}")
else:
    print("未调用模型：请在 .env 中配置 LLM_API_KEY（或 OPENAI_API_KEY）和 LLM_MODEL。")


问题：冲锋衣仅喷涂 DWR 涂层、未做全压胶处理，Listing 标题能否宣传为 100% Waterproof？

答案：
不能。根据法规，仅 DWR 处理的面料（无全压胶）不得标注“100% Waterproof”“Stormproof”“Rainproof”等字样。即使产品规格或质检报告宣称符合全防水条件，若实际未做全压胶处理，仍不得作此宣传。

依据来源：  
- 法规政策/FTC纺织标识法.md（4.2 禁止行为）
